# Notebook 48: Lifecycle

This notebook demonstrates agent lifecycle management using `multigen.lifecycle`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `AgentDependencyGraph` — register dependencies, compute affected workflows
- `SLAMonitor` — add SLA contracts, record metrics, detect violations
- `ShadowExecutor` — run primary + shadow agent, compare outputs
- `CapabilityRouter` — register capabilities at multiple versions, canary routing
- `AgentLifecycleManager` facade — register, shadow, deprecate, SLA checks

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.lifecycle`

In [ ]:
import time
import statistics
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

print('Imports ready.')


## Section 1 — AgentDependencyGraph

In [ ]:
class AgentDependencyGraph:
    """
    Tracks which workflows depend on which agents.
    workflow -> [agent, agent, ...]
    """

    def __init__(self):
        self._deps: Dict[str, List[str]] = {}   # workflow_id -> [agent_id]

    def register_dependency(self, workflow_id: str, agent_id: str):
        self._deps.setdefault(workflow_id, []).append(agent_id)

    def affected_workflows(self, agent_id: str) -> List[str]:
        """Return all workflows that depend on the given agent."""
        return [wf for wf, agents in self._deps.items() if agent_id in agents]

    def impact_report(self, agent_id: str) -> Dict:
        affected = self.affected_workflows(agent_id)
        return {
            'agent_id': agent_id,
            'affected_workflow_count': len(affected),
            'affected_workflows': affected,
        }


dep_graph = AgentDependencyGraph()

# Register 3 workflows and their agent dependencies
dep_graph.register_dependency('wf_report_gen',  'summariser')
dep_graph.register_dependency('wf_report_gen',  'formatter')
dep_graph.register_dependency('wf_qa_pipeline', 'summariser')
dep_graph.register_dependency('wf_qa_pipeline', 'retriever')
dep_graph.register_dependency('wf_email_digest','formatter')
dep_graph.register_dependency('wf_email_digest','summariser')

print('Impact report for "summariser":')
rpt = dep_graph.impact_report('summariser')
for k, v in rpt.items():
    print(f'  {k}: {v}')

print('\nImpact report for "retriever":')
rpt2 = dep_graph.impact_report('retriever')
for k, v in rpt2.items():
    print(f'  {k}: {v}')

print('\naffected_workflows("formatter"):', dep_graph.affected_workflows('formatter'))


## Section 2 — SLAMonitor

In [ ]:
@dataclass
class SLAContract:
    agent_id: str
    max_latency_ms: float
    min_success_rate: float   # 0–1


@dataclass
class SLAViolation:
    agent_id: str
    metric: str
    observed: float
    threshold: float

    def __str__(self):
        return (f'SLAViolation: agent={self.agent_id} metric={self.metric} '
                f'observed={self.observed:.3f} threshold={self.threshold:.3f}')


class SLAMonitor:
    def __init__(self):
        self._contracts: Dict[str, SLAContract] = {}
        self._latencies: Dict[str, List[float]] = {}
        self._successes: Dict[str, List[bool]] = {}

    def add_contract(self, contract: SLAContract):
        self._contracts[contract.agent_id] = contract

    def record(self, agent_id: str, latency_ms: float, success: bool):
        self._latencies.setdefault(agent_id, []).append(latency_ms)
        self._successes.setdefault(agent_id, []).append(success)

    def check_all(self) -> List[SLAViolation]:
        violations = []
        for agent_id, contract in self._contracts.items():
            lats = self._latencies.get(agent_id, [])
            succs = self._successes.get(agent_id, [])

            if lats:
                p95 = sorted(lats)[int(0.95 * len(lats))]
                if p95 > contract.max_latency_ms:
                    violations.append(SLAViolation(
                        agent_id=agent_id, metric='p95_latency_ms',
                        observed=p95, threshold=contract.max_latency_ms
                    ))
            if succs:
                sr = sum(succs) / len(succs)
                if sr < contract.min_success_rate:
                    violations.append(SLAViolation(
                        agent_id=agent_id, metric='success_rate',
                        observed=sr, threshold=contract.min_success_rate
                    ))
        return violations


monitor = SLAMonitor()
monitor.add_contract(SLAContract('summariser', max_latency_ms=200, min_success_rate=0.95))
monitor.add_contract(SLAContract('retriever',  max_latency_ms=100, min_success_rate=0.98))

# Simulate good metrics for retriever
for _ in range(20):
    monitor.record('retriever', latency_ms=60, success=True)

# Simulate degraded metrics for summariser
for i in range(20):
    monitor.record('summariser', latency_ms=150 + i * 10, success=(i < 18))  # 2 failures

print('SLA check results:')
violations = monitor.check_all()
if violations:
    for v in violations:
        print(f'  {v}')
else:
    print('  No violations.')


## Section 3 — ShadowExecutor

In [ ]:
@dataclass
class ShadowResult:
    input: Any
    primary_output: Any
    shadow_output: Any
    primary_latency_ms: float
    shadow_latency_ms: float
    agreement: bool

    def summary(self) -> str:
        return (
            f'agreement={self.agreement}  '
            f'primary_latency={self.primary_latency_ms:.1f}ms  '
            f'shadow_latency={self.shadow_latency_ms:.1f}ms'
        )


class ShadowExecutor:
    def __init__(self, primary: Callable, shadow: Callable):
        self.primary = primary
        self.shadow = shadow

    def run(self, input_data: Any) -> ShadowResult:
        t0 = time.perf_counter()
        primary_out = self.primary(input_data)
        primary_ms = (time.perf_counter() - t0) * 1000

        t1 = time.perf_counter()
        shadow_out = self.shadow(input_data)
        shadow_ms = (time.perf_counter() - t1) * 1000

        return ShadowResult(
            input=input_data,
            primary_output=primary_out,
            shadow_output=shadow_out,
            primary_latency_ms=round(primary_ms, 3),
            shadow_latency_ms=round(shadow_ms, 3),
            agreement=(primary_out == shadow_out),
        )


# Mock agents
def primary_agent(text: str) -> str:
    """Stable v1 agent."""
    time.sleep(0.001)   # simulate 1ms
    return f'[v1] Summary of: {text[:20]}'

def shadow_agent_same(text: str) -> str:
    """v2 agent with same output."""
    time.sleep(0.0005)
    return f'[v1] Summary of: {text[:20]}'  # same output

def shadow_agent_different(text: str) -> str:
    """v2 agent with different output."""
    time.sleep(0.0005)
    return f'[v2] Improved summary: {text[:20]}'


executor_agree = ShadowExecutor(primary_agent, shadow_agent_same)
executor_differ = ShadowExecutor(primary_agent, shadow_agent_different)

sample_input = 'The quarterly results exceeded analyst expectations significantly'

r1 = executor_agree.run(sample_input)
print(f'Shadow run (agreeing):     {r1.summary()}')
print(f'  primary_output : {r1.primary_output}')
print(f'  shadow_output  : {r1.shadow_output}')

r2 = executor_differ.run(sample_input)
print(f'\nShadow run (disagreeing):  {r2.summary()}')
print(f'  primary_output : {r2.primary_output}')
print(f'  shadow_output  : {r2.shadow_output}')


## Section 4 — CapabilityRouter

In [ ]:
@dataclass
class CapabilityVersion:
    version: str
    handler: Callable
    status: str = 'active'   # 'active', 'canary', 'deprecated'


class CapabilityRouter:
    def __init__(self):
        self._caps: Dict[str, List[CapabilityVersion]] = {}

    def register(self, capability: str, version: str, handler: Callable, status: str = 'active'):
        self._caps.setdefault(capability, []).append(
            CapabilityVersion(version=version, handler=handler, status=status)
        )

    def resolve(self, capability: str, prefer_canary: bool = False) -> CapabilityVersion:
        versions = self._caps.get(capability, [])
        if not versions:
            raise KeyError(f'No handler for capability: {capability}')
        if prefer_canary:
            canaries = [v for v in versions if v.status == 'canary']
            if canaries:
                return canaries[-1]
        active = [v for v in versions if v.status == 'active']
        if not active:
            raise RuntimeError(f'No active version for capability: {capability}')
        return active[-1]

    def deprecate(self, capability: str, version: str):
        for v in self._caps.get(capability, []):
            if v.version == version:
                v.status = 'deprecated'
                print(f'  Deprecated {capability}@{version}')

    def list_versions(self, capability: str) -> List[CapabilityVersion]:
        return self._caps.get(capability, [])


def summarise_v1(text: str) -> str:
    return f'[v1-summary] {text[:30]}'

def summarise_v2(text: str) -> str:
    return f'[v2-summary] {text[:30]} (improved)'


router = CapabilityRouter()
router.register('summarise', 'v1', summarise_v1, status='active')
router.register('summarise', 'v2', summarise_v2, status='canary')

print('All versions of "summarise":')
for cv in router.list_versions('summarise'):
    print(f'  {cv.version}: status={cv.status}')

# Resolve active (default)
active_ver = router.resolve('summarise')
print(f'\nResolved (active) -> {active_ver.version}')
print(f'Output: {active_ver.handler("The quarterly results exceeded expectations")}')

# Resolve canary
canary_ver = router.resolve('summarise', prefer_canary=True)
print(f'\nResolved (canary) -> {canary_ver.version}')
print(f'Output: {canary_ver.handler("The quarterly results exceeded expectations")}')

# Deprecate v1
print()
router.deprecate('summarise', 'v1')
print('All versions after deprecation:')
for cv in router.list_versions('summarise'):
    print(f'  {cv.version}: status={cv.status}')


## Section 5 — AgentLifecycleManager Facade

In [ ]:
class AgentLifecycleManager:
    """Facade combining all lifecycle primitives."""

    def __init__(self):
        self.dep_graph = AgentDependencyGraph()
        self.sla_monitor = SLAMonitor()
        self.router = CapabilityRouter()
        self._shadows: Dict[str, ShadowExecutor] = {}

    def register(self, agent_id: str, handler: Callable, version: str = 'v1',
                 sla: SLAContract = None):
        self.router.register(agent_id, version, handler, status='active')
        if sla:
            self.sla_monitor.add_contract(sla)
        print(f'[Lifecycle] Registered {agent_id}@{version}')

    def register_shadow(self, agent_id: str, primary: Callable, shadow: Callable):
        self._shadows[agent_id] = ShadowExecutor(primary, shadow)
        print(f'[Lifecycle] Shadow registered for {agent_id}')

    def shadow_call(self, agent_id: str, input_data: Any) -> ShadowResult:
        if agent_id not in self._shadows:
            raise KeyError(f'No shadow registered for {agent_id}')
        result = self._shadows[agent_id].run(input_data)
        return result

    def deprecate(self, agent_id: str, version: str):
        self.router.deprecate(agent_id, version)

    def check_sla_violations(self) -> List[SLAViolation]:
        return self.sla_monitor.check_all()


mgr = AgentLifecycleManager()

# Register agents
mgr.register('summariser', summarise_v1, version='v1',
             sla=SLAContract('summariser', max_latency_ms=300, min_success_rate=0.95))
mgr.register('classifier', lambda t: 'positive', version='v1')

# Register shadow for summariser
mgr.register_shadow('summariser', summarise_v1, summarise_v2)

# Run shadow comparison
sr = mgr.shadow_call('summariser', 'Revenue grew 20% year over year')
print(f'\nShadow call result: {sr.summary()}')
print(f'  primary: {sr.primary_output}')
print(f'  shadow : {sr.shadow_output}')

# Record some SLA metrics
for i in range(10):
    mgr.sla_monitor.record('summariser', latency_ms=280 + i * 5, success=True)

# Check SLA
print('\nSLA violations:')
violations = mgr.check_sla_violations()
if violations:
    for v in violations:
        print(f'  {v}')
else:
    print('  None.')

# Deprecate v1
print()
mgr.deprecate('summariser', 'v1')
